# PI05 Policy Server 客户端仿真

与本地 notebook 推理方式一致：**客户端几乎不做数据处理**，直接发送 `dict(ds[i])`；  
server 端调用 `preprocess(frame)` 完成 rename / batch / normalize / tokenize。

## 数据流

| 阶段 | 谁负责 | 做什么 |
|------|--------|--------|
| 客户端 | 你 | `LeRobotDataset[i]` → msgpack 序列化（tensor→numpy） |
| server `preprocess` | server | 与本地 `make_pre_post_processors` 相同流水线 |
| server `policy` | server | `predict_action_chunk` |
| server `postprocess` | server | action 反归一化 |

## 前置条件

先启动 server，再运行本 notebook。

In [1]:
# --- 0) 路径与环境 ---
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np

# 本 notebook 所在目录 & my_vla 根目录
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "scripts":
    NOTEBOOK_DIR = Path("/vla/my_vla/scripts")
MY_VLA_ROOT = NOTEBOOK_DIR.parent
LEROBOT_SRC = MY_VLA_ROOT / "src" / "lerobot" / "src"

for p in (str(LEROBOT_SRC), str(NOTEBOOK_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

# 数据集路径
DATA_ROOT = Path("/vla/.data")
ADJUST_BOTTLE_ROOT = DATA_ROOT / "adjust_bottle"
TEST_ROOT = DATA_ROOT / "test"

# WebSocket server 地址（与 serve_pi05_policy.py --host/--port 一致）
WS_HOST = "127.0.0.1"
WS_PORT = 8000

print("LeRobot 源码:", LEROBOT_SRC)
print("adjust_bottle:", ADJUST_BOTTLE_ROOT, "exists=", ADJUST_BOTTLE_ROOT.is_dir())
print("test:", TEST_ROOT, "exists=", TEST_ROOT.is_dir())
print("Server:", f"ws://{WS_HOST}:{WS_PORT}")

LeRobot 源码: /vla/my_vla/src/lerobot/src
adjust_bottle: /vla/.data/adjust_bottle exists= True
test: /vla/.data/test exists= True
Server: ws://127.0.0.1:8000


## 1) 客户端：只做「序列化 + 发送」

与本地 `preprocess(ds[i])` 对比：**client 不做 normalize/tokenize**，只把 tensor 转成 numpy 以便 msgpack 传输。

```python
result = send_frame(client, ds_adjust[0], reset=True)
```

In [2]:
# --- 1) msgpack 客户端 + send_frame（唯一需要的封装）---
import functools
import logging

import msgpack
import websockets.sync.client

logging.basicConfig(level=logging.INFO)


def _pack_array(obj):
    if isinstance(obj, (np.ndarray, np.generic)) and obj.dtype.kind in ("V", "O", "c"):
        raise ValueError(f"Unsupported dtype: {obj.dtype}")
    if isinstance(obj, np.ndarray):
        return {b"__ndarray__": True, b"data": obj.tobytes(), b"dtype": obj.dtype.str, b"shape": obj.shape}
    if isinstance(obj, np.generic):
        return {b"__npgeneric__": True, b"data": obj.item(), b"dtype": obj.dtype.str}
    return obj


def _unpack_array(obj):
    if isinstance(obj, dict) and b"__ndarray__" in obj:
        return np.ndarray(buffer=obj[b"data"], dtype=np.dtype(obj[b"dtype"]), shape=obj[b"shape"])
    if isinstance(obj, dict) and b"__npgeneric__" in obj:
        return np.dtype(obj[b"dtype"]).type(obj[b"data"])
    return obj


_packer = msgpack.Packer(default=_pack_array)
_unpack = functools.partial(msgpack.unpackb, object_hook=_unpack_array)


class WebsocketClientPolicy:
    def __init__(self, host="127.0.0.1", port=8000):
        self._uri = host if host.startswith("ws") else f"ws://{host}:{port}"
        self._ws, self.metadata = self._connect()

    def _connect(self):
        while True:
            try:
                ws = websockets.sync.client.connect(self._uri, compression=None, max_size=None)
                return ws, _unpack(ws.recv())
            except OSError as exc:
                logging.info("等待 server... (%s)", exc)
                time.sleep(2)

    def infer(self, payload: dict) -> dict:
        self._ws.send(_packer.pack(payload))
        resp = self._ws.recv()
        if isinstance(resp, str):
            raise RuntimeError(f"Server error:\n{resp}")
        return _unpack(resp)

    def close(self):
        try:
            self._ws.close()
        except Exception:
            pass


def send_frame(client, frame: dict, *, timestep: int = 0, reset: bool = False) -> dict:
    """把 LeRobotDataset 一帧发给 server（等同本地 preprocess 的输入）。"""
    payload = {
        k: (v.detach().cpu().numpy() if hasattr(v, "detach") else v)
        for k, v in frame.items()
    }
    payload["timestep"] = int(timestep)
    payload["reset"] = bool(reset)
    return client.infer(payload)

## 2) 连接 Server

连接成功后，server 会先下发 `metadata`（checkpoint 路径、期望 image key、action 维等），可用于核对客户端是否对齐。

In [3]:
client = WebsocketClientPolicy(host=WS_HOST, port=WS_PORT)
print(json.dumps(client.metadata, indent=2, ensure_ascii=False))

INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection refused)
INFO:root:等待 server... ([Errno 111] Connection r

{
  "model_type": "pi05",
  "deployment": "lerobot_generic_websocket_server",
  "checkpoint_dir": "/vla/my_vla/policy/checkpoints/2026-06-09/11-45-55_myvla_adjust_bottle/checkpoints/010000/pretrained_model",
  "device": "cuda",
  "dtype": "torch.float32",
  "infer_horizon": 16,
  "chunk_size": 50,
  "supports_chunking": true,
  "default_prompt": "Execute the task.",
  "expected_image_keys": [
    "observation.images.cam_high",
    "observation.images.cam_left_wrist",
    "observation.images.cam_right_wrist"
  ],
  "expected_input_keys": [
    "observation.images.cam_high",
    "observation.images.cam_left_wrist",
    "observation.images.cam_right_wrist",
    "observation.state"
  ],
  "config_state_dim": 32,
  "norm_state_dim": 14,
  "expected_action_dim": 14,
  "rename_map_override": {},
  "tokenizer_path": "/mnt/workspace/luyi/.cache/huggingface/hub/models--google--paligemma-3b-pt-224/snapshots/35e4f46485b4d07967e7e9935bc3786aad50687c",
  "notes": {
    "client_payload": "推荐发送 dict(L

## 3) 数据集 A：`adjust_bottle`

键名已与 checkpoint 对齐，一行发送即可：`send_frame(client, ds[i])`

In [4]:
# --- 3a) 加载 adjust_bottle ---
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds_adjust = LeRobotDataset(repo_id="adjust_bottle", root=str(ADJUST_BOTTLE_ROOT))

episode_index = 0
from_idx = ds_adjust.meta.episodes["dataset_from_index"][episode_index]
frame_idx = int(from_idx)
frame_adjust = ds_adjust[frame_idx]

print("keys:", sorted(frame_adjust.keys()))
print("task:", frame_adjust["task"])
print("state:", frame_adjust["observation.state"].shape, frame_adjust["observation.state"].dtype)
print("cam_high:", frame_adjust["observation.images.cam_high"].shape)

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


keys: ['action', 'episode_index', 'frame_index', 'index', 'observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist', 'observation.state', 'task', 'task_index', 'timestamp']
task: Use the left arm to pick up the narrow-necked bottle upright from the table.
state: torch.Size([14]) torch.float32
cam_high: torch.Size([3, 480, 640])


In [5]:
# --- 3b) 发送 ds[i] ---
result_adjust = send_frame(client, frame_adjust, timestep=0, reset=True)
print("actions shape:", result_adjust["actions"].shape)
print("action[0]:", np.round(result_adjust["action"], 4))
print("server_timing:", result_adjust.get("server_timing"))

actions shape: (16, 14)
action[0]: [-7.010e-02  5.457e-01  4.454e-01 -9.930e-02 -8.000e-04  4.460e-02
  8.276e-01  0.000e+00 -0.000e+00  0.000e+00 -0.000e+00  0.000e+00
 -0.000e+00  1.000e+00]
server_timing: {'infer_ms': 1778.7518668919802}


## 4) 数据集 B：`test`

FRANKA 数据，**相机 key 与 checkpoint 不同**。只需做键名对齐（等同你 notebook 里的 `rename_map`），仍不需要 normalize。

| 数据集 key | checkpoint key |
|-----------|----------------|
| `observation.images.robot0_agentview_left_image` | `observation.images.cam_high` |
| `observation.images.robot0_eye_in_hand_image` | `observation.images.cam_left_wrist` |
| `observation.images.robot0_agentview_right_image` | `observation.images.cam_right_wrist` |

此外 `test` 的 **state=9、action=12**，与本 checkpoint 的 normalizer stats **14 维**不一致，需在 client 末维 **补 0**（仅用于链路测试，语义不对齐）。

也可在 server 启动时传 `--rename_map` JSON，则客户端可原样发送图像 key。

In [6]:
# --- 4a) 加载 test ---
ds_test = LeRobotDataset(repo_id="test", root=str(TEST_ROOT))

frame_idx = int(ds_test.meta.episodes["dataset_from_index"][0])
frame_test = ds_test[frame_idx]

print("keys:", sorted(frame_test.keys()))
print("task:", frame_test["task"])
print("state:", frame_test["observation.state"].shape)
for k in sorted(frame_test.keys()):
    if "image" in k:
        print(k, frame_test[k].shape)

keys: ['action', 'episode_index', 'frame_index', 'index', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image', 'observation.state', 'task', 'task_index', 'timestamp']
task: robot manipulation task
state: torch.Size([9])
observation.images.robot0_agentview_left_image torch.Size([3, 224, 224])
observation.images.robot0_agentview_right_image torch.Size([3, 224, 224])
observation.images.robot0_eye_in_hand_image torch.Size([3, 224, 224])


In [7]:
# --- 4b) test：改 key + 补维后 send_frame ---
TEST_RENAME = {
    "observation.images.robot0_agentview_left_image": "observation.images.cam_high",
    "observation.images.robot0_eye_in_hand_image": "observation.images.cam_left_wrist",
    "observation.images.robot0_agentview_right_image": "observation.images.cam_right_wrist",
}

# checkpoint normalizer stats 的 state/action 维数（本 ckpt 均为 14；config 里 state=32 是模型内部 pad）
NORM_VECTOR_DIM = client.metadata["expected_action_dim"]


def _as_numpy(v):
    return v.detach().cpu().numpy() if hasattr(v, "detach") else np.asarray(v)


def pad_last_dim(v, target_dim: int):
    """末维零填充到 target_dim，供 preprocess normalize 与 stats 对齐。"""
    arr = _as_numpy(v)
    cur = arr.shape[-1]
    if cur == target_dim:
        return arr
    if cur > target_dim:
        return arr[..., :target_dim]
    pad = [(0, 0)] * (arr.ndim - 1) + [(0, target_dim - cur)]
    return np.pad(arr, pad)


frame_test_aligned = dict(frame_test)
for old, new in TEST_RENAME.items():
    if old in frame_test_aligned:
        frame_test_aligned[new] = frame_test_aligned.pop(old)

frame_test_aligned["observation.state"] = pad_last_dim(
    frame_test_aligned["observation.state"], NORM_VECTOR_DIM
)
if "action" in frame_test_aligned:
    frame_test_aligned["action"] = pad_last_dim(frame_test_aligned["action"], NORM_VECTOR_DIM)

print(
    "aligned state/action:",
    frame_test_aligned["observation.state"].shape,
    frame_test_aligned.get("action", np.array([])).shape,
)

result_test = send_frame(client, frame_test_aligned, reset=True)
print("actions shape:", result_test["actions"].shape)
print("action[0]:", np.round(result_test["action"], 4))
print("server_timing:", result_test.get("server_timing"))

aligned state/action: (14,) (14,)
actions shape: (16, 14)
action[0]: [-0.0442  0.0074  0.4291 -1.1684  0.1454 -0.0453  0.0441  0.      0.
  0.      0.      0.      0.      1.    ]
server_timing: {'infer_ms': 381.15261076018214, 'prev_total_ms': 5458.403239026666}


## 5) 可选：连续多帧推理（adjust_bottle）

演示在同一 episode 内连续请求多帧。第一帧 `reset=True`，后续帧 `reset=False`。

In [8]:
# --- 5) 连续 3 帧 ---
from_idx = int(ds_adjust.meta.episodes["dataset_from_index"][0])
to_idx = int(ds_adjust.meta.episodes["dataset_to_index"][0])
for i, idx in enumerate(range(from_idx, min(from_idx + 3, to_idx))):
    r = send_frame(client, ds_adjust[idx], timestep=i, reset=(i == 0))
    print(f"frame={idx} action[:4]={np.round(r['action'][:4], 3)}")

frame=0 action[:4]=[ 0.004  0.    -0.022  0.03 ]
frame=1 action[:4]=[ 0.013 -0.019 -0.039  0.016]
frame=2 action[:4]=[ 0.009 -0.048 -0.022  0.03 ]


In [9]:
# --- 清理连接 ---
client.close()
print("done")

done


In [11]:
result_test['actions'].shape

(16, 14)

``` mermaid
flowchart LR
  A[client frame] --> B[normalizer<br/>维数须 = stats 维]
  B --> C[pi05_prepare<br/>state pad → 32<br/>写入 prompt]
  C --> D[tokenizer + 模型<br/>action 内部 32 维]
  D --> E[predict_action_chunk<br/>action 截断 → 14]
  E --> F[postprocessor<br/>反归一化 14 维]
```